# LAB 05 — Sequence Models: RNN, LSTM, and GRU

Môn học: Xử lý ngôn ngữ tự nhiên và ứng dụng

Notebook này triển khai toàn bộ quy trình xây dựng, huấn luyện và đánh giá các mô hình chuỗi (Simple RNN, LSTM, GRU) cùng mô hình cơ sở (TF-IDF + Logistic Regression) trên tập dữ liệu IMDb Movie Reviews.

## 1. Khởi tạo môi trường và nạp các thư viện

In [1]:
import os
import re
import time
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

## 2. Nạp dữ liệu IMDb và tiền xử lý văn bản

In [2]:
dataset_path = '../dataset/IMDB Dataset.csv'
if not os.path.exists(dataset_path):
    dataset_path = 'd:/Hus/NLP/Lab5/dataset/IMDB Dataset.csv'

dataframe = pd.read_csv(dataset_path)
dataframe['clean_text'] = dataframe['review'].apply(lambda s: re.sub(r'<[^>]+>', ' ', s).lower().strip())
dataframe['word_count'] = dataframe['clean_text'].apply(lambda s: len(s.split()))
dataframe['label'] = dataframe['sentiment'].map({'positive': 1, 'negative': 0})

sample_dataframe = dataframe.sample(n=10000, random_state=42).reset_index(drop=True)
train_dataframe, test_dataframe = train_test_split(sample_dataframe, test_size=0.3, random_state=42, stratify=sample_dataframe['label'])
val_dataframe, test_dataframe = train_test_split(test_dataframe, test_size=0.5, random_state=42, stratify=test_dataframe['label'])

print(f"Train size: {len(train_dataframe)}, Val size: {len(val_dataframe)}, Test size: {len(test_dataframe)}")

Train size: 7000, Val size: 1500, Test size: 1500


## 3. Baseline: TF-IDF + Logistic Regression (LAB 04)

In [3]:
start_baseline_time = time.time()
tfidf_vectorizer = TfidfVectorizer(max_features=10000, stop_words='english')
train_tfidf_matrix = tfidf_vectorizer.fit_transform(train_dataframe['clean_text'])
test_tfidf_matrix = tfidf_vectorizer.transform(test_dataframe['clean_text'])

baseline_classifier = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
baseline_classifier.fit(train_tfidf_matrix, train_dataframe['label'])
baseline_training_duration = time.time() - start_baseline_time

baseline_predictions = baseline_classifier.predict(test_tfidf_matrix)
baseline_accuracy = accuracy_score(test_dataframe['label'], baseline_predictions)
baseline_f1 = f1_score(test_dataframe['label'], baseline_predictions)

print(f"Baseline Accuracy: {baseline_accuracy:.4f}, F1: {baseline_f1:.4f}, Time: {baseline_training_duration:.2f}s")

Baseline Accuracy: 0.8700, F1: 0.8741, Time: 1.23s


## 4. Xây dựng Vocabulary và PyTorch Dataset

In [4]:
token_frequency_counter = Counter()
for text in train_dataframe['clean_text']:
    token_frequency_counter.update(text.split())

vocabulary_mapping = {'<PAD>': 0, '<UNK>': 1}
for token, count in token_frequency_counter.most_common(9998):
    vocabulary_mapping[token] = len(vocabulary_mapping)
vocabulary_capacity = len(vocabulary_mapping)

def encode_text_sequence(text_input, sequence_cutoff=100):
    tokens = text_input.split()
    identifier_list = [vocabulary_mapping.get(t, 1) for t in tokens[:sequence_cutoff]]
    if len(identifier_list) < sequence_cutoff:
        identifier_list += [0] * (sequence_cutoff - len(identifier_list))
    return identifier_list

class IMDBSequenceDataset(Dataset):
    def __init__(self, data_frame, sequence_length=100):
        self.token_sequences = [encode_text_sequence(t, sequence_length) for t in data_frame['clean_text']]
        self.target_labels = data_frame['label'].values
    def __len__(self):
        return len(self.token_sequences)
    def __getitem__(self, position_index):
        return torch.tensor(self.token_sequences[position_index], dtype=torch.long), torch.tensor(self.target_labels[position_index], dtype=torch.long)

## 5. Định nghĩa kiến trúc các mô hình Recurrent (RNN, LSTM, GRU)

In [5]:
class RecurrentTextClassifier(nn.Module):
    def __init__(self, recurrent_architecture='lstm', vocab_size=10000, embedding_dim=64, hidden_state_dim=64, output_classes=2):
        super(RecurrentTextClassifier, self).__init__()
        self.recurrent_architecture = recurrent_architecture
        self.embedding_layer = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        if recurrent_architecture == 'rnn':
            self.recurrent_cell = nn.RNN(embedding_dim, hidden_state_dim, batch_first=True)
        elif recurrent_architecture == 'lstm':
            self.recurrent_cell = nn.LSTM(embedding_dim, hidden_state_dim, batch_first=True)
        elif recurrent_architecture == 'gru':
            self.recurrent_cell = nn.GRU(embedding_dim, hidden_state_dim, batch_first=True)
        self.classification_head = nn.Linear(hidden_state_dim, output_classes)

    def forward(self, input_sequence_tensor):
        embedded_vectors = self.embedding_layer(input_sequence_tensor)
        if self.recurrent_architecture == 'lstm':
            temporal_outputs, (final_hidden, final_cell) = self.recurrent_cell(embedded_vectors)
        else:
            temporal_outputs, final_hidden = self.recurrent_cell(embedded_vectors)
        prediction_logits = self.classification_head(temporal_outputs[:, -1, :])
        return prediction_logits

## 6. Pipeline huấn luyện và đánh giá mô hình

In [6]:
def execute_model_lifecycle(architecture_name, max_sequence_length=100, total_training_epochs=5):
    training_dataset = IMDBSequenceDataset(train_dataframe, max_sequence_length)
    validation_dataset = IMDBSequenceDataset(val_dataframe, max_sequence_length)
    testing_dataset = IMDBSequenceDataset(test_dataframe, max_sequence_length)

    train_data_loader = DataLoader(training_dataset, batch_size=64, shuffle=True)
    val_data_loader = DataLoader(validation_dataset, batch_size=64, shuffle=False)
    test_data_loader = DataLoader(testing_dataset, batch_size=64, shuffle=False)

    neural_network = RecurrentTextClassifier(recurrent_architecture=architecture_name, vocab_size=vocabulary_capacity, embedding_dim=64, hidden_state_dim=64)
    optimization_algorithm = torch.optim.Adam(neural_network.parameters(), lr=0.001)
    loss_objective_function = nn.CrossEntropyLoss()

    training_loss_history = []
    validation_loss_history = []

    timer_start = time.time()
    for current_epoch in range(total_training_epochs):
        neural_network.train()
        cumulative_epoch_loss = 0.0
        for sequence_batch, label_batch in train_data_loader:
            optimization_algorithm.zero_grad()
            batch_predictions = neural_network(sequence_batch)
            calculated_loss = loss_objective_function(batch_predictions, label_batch)
            calculated_loss.backward()
            optimization_algorithm.step()
            cumulative_epoch_loss += calculated_loss.item() * len(label_batch)
        training_loss_history.append(cumulative_epoch_loss / len(train_dataframe))

        neural_network.eval()
        cumulative_val_loss = 0.0
        with torch.no_grad():
            for sequence_batch, label_batch in val_data_loader:
                val_predictions = neural_network(sequence_batch)
                val_batch_loss = loss_objective_function(val_predictions, label_batch)
                cumulative_val_loss += val_batch_loss.item() * len(label_batch)
        validation_loss_history.append(cumulative_val_loss / len(val_dataframe))

    elapsed_duration = time.time() - timer_start

    neural_network.eval()
    collected_predictions = []
    collected_ground_truth = []
    with torch.no_grad():
        for sequence_batch, label_batch in test_data_loader:
            logits_output = neural_network(sequence_batch)
            class_decisions = torch.argmax(logits_output, dim=1)
            collected_predictions.extend(class_decisions.numpy())
            collected_ground_truth.extend(label_batch.numpy())

    measured_accuracy = accuracy_score(collected_ground_truth, collected_predictions)
    measured_precision = precision_score(collected_ground_truth, collected_predictions)
    measured_recall = recall_score(collected_ground_truth, collected_predictions)
    measured_f1 = f1_score(collected_ground_truth, collected_predictions)
    trainable_parameter_count = sum(p.numel() for p in neural_network.parameters() if p.requires_grad)

    return {
        'model': neural_network,
        'params': trainable_parameter_count,
        'duration': elapsed_duration,
        'acc': measured_accuracy,
        'prec': measured_precision,
        'rec': measured_recall,
        'f1': measured_f1,
        'train_loss': training_loss_history,
        'val_loss': validation_loss_history,
        'predictions': collected_predictions
    }

## 7. Thực nghiệm so sánh RNN, LSTM và GRU

In [7]:
rnn_experimental_outcome = execute_model_lifecycle('rnn')
lstm_experimental_outcome = execute_model_lifecycle('lstm')
gru_experimental_outcome = execute_model_lifecycle('gru')

print(f"Simple RNN  — Accuracy: {rnn_experimental_outcome['acc']:.4f}, F1: {rnn_experimental_outcome['f1']:.4f}, Params: {rnn_experimental_outcome['params']}")
print(f"LSTM Model  — Accuracy: {lstm_experimental_outcome['acc']:.4f}, F1: {lstm_experimental_outcome['f1']:.4f}, Params: {lstm_experimental_outcome['params']}")
print(f"GRU Model   — Accuracy: {gru_experimental_outcome['acc']:.4f}, F1: {gru_experimental_outcome['f1']:.4f}, Params: {gru_experimental_outcome['params']}")

Simple RNN  — Accuracy: 0.4920, F1: 0.4694, Params: 648450
LSTM Model  — Accuracy: 0.5220, F1: 0.5139, Params: 673410
GRU Model   — Accuracy: 0.6667, F1: 0.6676, Params: 665090


## 8. Ứng dụng suy luận cảm xúc trên các mẫu câu cụ thể (Mục 26)

In [8]:
def predict_sentiment_for_raw_review(model_instance, raw_text_sentence):
    model_instance.eval()
    cleaned_sentence = re.sub(r'<[^>]+>', ' ', raw_text_sentence).lower().strip()
    token_ids = encode_text_sequence(cleaned_sentence, sequence_cutoff=100)
    input_tensor = torch.tensor([token_ids], dtype=torch.long)
    with torch.no_grad():
        network_logits = model_instance(input_tensor)
        probability_distribution = torch.softmax(network_logits, dim=1)
        predicted_class_index = torch.argmax(network_logits, dim=1).item()
    sentiment_label = 'Positive' if predicted_class_index == 1 else 'Negative'
    assigned_confidence = probability_distribution[0, predicted_class_index].item()
    return sentiment_label, assigned_confidence

test_sample_queries = [
    'I really enjoyed this movie.',
    'I thought the movie would be terrible, but it was surprisingly good.',
    'This was not a good film at all.',
    'The actors tried their best, but the script was completely uninspired and dull.'
]

for sample_query in test_sample_queries:
    label, confidence = predict_sentiment_for_raw_review(gru_experimental_outcome['model'], sample_query)
    print(f"Text: '{sample_query}' -> Sentiment: {label} (Confidence: {confidence:.4f})")

Text: 'I really enjoyed this movie.' -> Sentiment: Negative (Confidence: 0.7733)
Text: 'I thought the movie would be terrible, but it was surprisingly good.' -> Sentiment: Positive (Confidence: 0.8629)
Text: 'This was not a good film at all.' -> Sentiment: Negative (Confidence: 0.7754)
Text: 'The actors tried their best, but the script was completely uninspired and dull.' -> Sentiment: Negative (Confidence: 0.7756)
